In [ ]:
from webdav3.client import Client
from dotenv import load_dotenv
load_dotenv()
import os
import pandas as pd
import xml.etree.ElementTree as ET
import regex as re
from tqdm import tqdm
import json
import requests
from requests.auth import HTTPBasicAuth
from sqlalchemy import create_engine, MetaData, Table, select, insert
from sqlalchemy.exc import SQLAlchemyError



In [21]:
def webdav_login(server_url, username, password):
    try:
        # connect 2 webdav server
        client = Client({
            'webdav_hostname': server_url,
            'webdav_login': username,
            'webdav_password': password
        })

       #check connection
        if client.check():
            return client
        else:
            print("Wrong login data.")
            return None
    except Exception as e:
        print(f"Exception: {e}")
        return None
    

def get_meta(client, path, server_url):
    url=f"{server_url}{path}"
    # PROPFIND Anfrage senden
    response = client.session.request(
        method="PROPFIND",
        url=url,
        headers={'Depth': '1'},
        data="""<?xml version="1.0" encoding="utf-8"?>
    <d:propfind xmlns:d="DAV:" xmlns:oc="http://owncloud.org/ns">
        <d:prop>
            <oc:id/>
            <oc:fileid/>
        </d:prop>
    </d:propfind>"""
    )

    # Parse XML response
    root = ET.fromstring(response.text)
    ns = {'oc': 'http://owncloud.org/ns'}
    fileid = root.find('.//oc:fileid', ns).text if root.find('.//oc:fileid', ns) is not None else None
    id = root.find('.//oc:id', ns).text if root.find('.//oc:id', ns) is not None else None
    return id, fileid

def folder_to_dict(path, client):
    entries = client.list(path)[1:]  # skip the listing of the directory itself
    children = {}
    for entry in entries:
        full_entry_path = path + entry
        if entry.endswith("/"):
            # recurse into subfolder
            children[entry] = folder_to_dict(full_entry_path, client)
        else:
            # file
            children[entry] = entry
    return children

def folder_to_dict_w_meta_tqdm(path, client, server_url):
    entries = client.list(path)[1:]  # skip the listing of the directory itself
    children = {}
    for entry in tqdm(entries):
        full_entry_path = path + entry
        if entry.endswith("/"):
            # recurse into subfolder
            children[entry] = folder_to_dict_w_meta(full_entry_path, client, server_url)
        else:
            # file: call file_id and store the response
            id, fileid = get_meta(client, full_entry_path, server_url)
            children[entry] = {"name": entry, "id": id, "fileid": fileid, "path": full_entry_path}
    return children


def folder_to_dict_w_meta(path, client, server_url):
    entries = client.list(path)[1:]  # skip the listing of the directory itself
    children = {}
    for entry in entries:
        full_entry_path = path + entry
        if entry.endswith("/"):
            # recurse into subfolder
            children[entry] = folder_to_dict_w_meta(full_entry_path, client, server_url)
        else:
            # file: call file_id and store the response
            id, fileid = get_meta(client, full_entry_path, server_url)
            children[entry] = {"name": entry, "id": id, "fileid": fileid, "path": full_entry_path}
    return children


In [22]:
load_dotenv()
#####################
DB_HOST = os.getenv("DB_HOST")
NC_ACC = os.getenv("NC_ACC")
NC_PASS = os.getenv("NC_PASS")


In [ ]:

server_url = f'''http://{DB_HOST}:8080/remote.php/dav/files/{NC_ACC}'''
username = NC_ACC
password = NC_PASS
path =  "/Bre/Artwork/"

client = webdav_login(server_url, username, password)

if client:
    print("client connected")
    root_dict = {path.strip("/").split("/")[-1]: folder_to_dict_w_meta_tqdm(path, client, server_url)}



client connected


100%|██████████| 10/10 [03:02<00:00, 18.23s/it]


In [32]:
def mk_preview_urls(file_id):
    preview_url = f'''/core/preview?fileId={file_id}&'''+'{prevsize}'
    return preview_url

def flatten_dict_to_list(data):
    result_list = []
    
    def process_dict(content):
        for key, val in content.items():
            # If it's a folder entry
            if key.endswith("/"):
                if isinstance(val, dict):
                    process_dict(val)
            # If it's a file entry with metadata
            elif isinstance(val, dict) and 'name' in val:
                result_list.append(val)
    
    # Start with the root content
    if len(data) == 1:
        first_val = next(iter(data.values()))
        content = first_val
    else:
        content = data
        
    process_dict(content)
    return result_list


In [ ]:
df_index = pd.DataFrame(flatten_dict_to_list(root_dict))
df_index['preview_url'] = df_index.fileid.apply(lambda x: mk_preview_urls(x))

In [50]:
prev_size = 1080
for index, row in df_index.head(10).iterrows():
    prev_url = row['preview_url']
    prev_url = f'''http://{DB_HOST}:8080{prev_url.replace('{prevsize}', f'''x={prev_size}&y={prev_size}''')}'''
    print(prev_url)

http://192.168.0.150:8080/core/preview?fileId=2158&x=1080&y=1080
http://192.168.0.150:8080/core/preview?fileId=2153&x=1080&y=1080
http://192.168.0.150:8080/core/preview?fileId=2150&x=1080&y=1080
http://192.168.0.150:8080/core/preview?fileId=2141&x=1080&y=1080
http://192.168.0.150:8080/core/preview?fileId=2149&x=1080&y=1080
http://192.168.0.150:8080/core/preview?fileId=2160&x=1080&y=1080
http://192.168.0.150:8080/core/preview?fileId=2592&x=1080&y=1080
http://192.168.0.150:8080/core/preview?fileId=2576&x=1080&y=1080
http://192.168.0.150:8080/core/preview?fileId=2594&x=1080&y=1080
http://192.168.0.150:8080/core/preview?fileId=2152&x=1080&y=1080


In [52]:
def create_db_connection():
    load_dotenv()
    DB_HOST = os.getenv("DB_HOST")
    DB_NAME = os.getenv("DB_NAME")
    DB_USER = os.getenv("DB_USER")
    DB_PASSWORD = os.getenv("DB_PASSWORD")
    engine = create_engine('postgresql+pg8000://'+DB_USER+':'+DB_PASSWORD+'@'+DB_HOST+':5432/'+DB_NAME)
    return engine


In [56]:
df_index.to_sql('bre_advance_index', if_exists='replace', index=False,con=create_db_connection())

1517